In [5]:
import os
import sys
import numpy as np
import pandas as pd
from pymilvus import MilvusClient, DataType

CLIP_DIR = "data/clip_features"
CSV_DIR = "data/csv_metadata"
MILVUS_URI = "./milvus_demo.db"     
COLLECTION_NAME = "clip_keyframes"

def normalize(vectors: np.ndarray) -> np.ndarray:
    """Chuẩn hóa L2 cho mảng vectors."""
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms[norms == 0] = 1e-12
    return vectors / norms

def create_collection(client: MilvusClient, dim: int, fresh: bool = False):
    """Khởi tạo collection với số chiều vector được truyền động."""
    if client.has_collection(COLLECTION_NAME):
        if fresh:
            print(f"[-] Xóa collection cũ: {COLLECTION_NAME}")
            client.drop_collection(COLLECTION_NAME)
        else:
            return

    schema = client.create_schema(auto_id=True, enable_dynamic_field=False)
    schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
    schema.add_field(field_name="video_id", datatype=DataType.VARCHAR, max_length=256)
    schema.add_field(field_name="frame_id", datatype=DataType.INT64)
    schema.add_field(field_name="embedding", datatype=DataType.FLOAT_VECTOR, dim=dim)

    # Cấu hình Index: Dùng FLAT cho Milvus Lite local để không đòi faiss-cpu
    index_params = client.prepare_index_params()
    index_params.add_index(
        field_name="embedding",
        index_type="FLAT",          # Đổi thành "HNSW" nếu chạy server Milvus lớn
        metric_type="IP",           # Inner Product (Cosine similarity khi vector đã L2-normalized)
        params={},
    )

    client.create_collection(
        collection_name=COLLECTION_NAME,
        schema=schema,
        index_params=index_params,
    )
    print(f"[+] Đã tạo collection mới: {COLLECTION_NAME} (dim={dim})")

def get_existing_videos(client: MilvusClient) -> set:
    """Lấy danh sách video_id đã có trong collection để tránh nạp trùng."""
    existing = set()
    try:
        iterator = client.query_iterator(
            collection_name=COLLECTION_NAME,
            filter="",
            output_fields=["video_id"],
            batch_size=1000,
        )
        while True:
            batch = iterator.next()
            if not batch:
                break
            existing.update(row["video_id"] for row in batch)
        iterator.close()
    except Exception:
        pass
    return existing

def insert_video(client: MilvusClient, npy_file: str) -> int:
    """Nạp vector và metadata của 1 video vào Milvus."""
    video_id = os.path.splitext(npy_file)[0]
    npy_path = os.path.join(CLIP_DIR, npy_file)
    csv_path = os.path.join(CSV_DIR, f"{video_id}.csv")

    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Không tìm thấy CSV tương ứng: {csv_path}")

    # Đọc vector
    features = np.load(npy_path).astype(np.float32)
    if features.ndim == 1:
        features = features.reshape(1, -1)
    
    # Đọc CSV metadata và xử lý frame_idx linh hoạt
    df = pd.read_csv(csv_path)
    if "frame_idx" in df.columns:
        frame_ids = df["frame_idx"].astype(int).tolist()
    elif "frame_id" in df.columns:
        frame_ids = df["frame_id"].astype(int).tolist()
    else:
        frame_ids = list(range(1, len(df) + 1))

    if len(features) != len(frame_ids):
        raise ValueError(
            f"{video_id}: Lệch kích thước ({len(features)} vectors != {len(frame_ids)} frames)"
        )

    features = normalize(features)

    rows = [
        {
            "video_id": video_id,
            "frame_id": frame_ids[i],
            "embedding": features[i].tolist(),
        }
        for i in range(len(features))
    ]

    result = client.insert(collection_name=COLLECTION_NAME, data=rows)
    print(f"  + {video_id}: Đã nạp {result['insert_count']} vectors")
    return result["insert_count"]

def run(mode: str = "build"):
    if not os.path.exists(CLIP_DIR):
        raise FileNotFoundError(f"Thư mục '{CLIP_DIR}' không tồn tại.")

    npy_files = sorted(f for f in os.listdir(CLIP_DIR) if f.endswith(".npy"))
    if not npy_files:
        raise RuntimeError(f"Không tìm thấy file .npy nào trong thư mục '{CLIP_DIR}'")

    # Tự động lấy số chiều vector từ file npy đầu tiên
    first_npy = np.load(os.path.join(CLIP_DIR, npy_files[0]))
    dim = first_npy.shape[-1]

    client = MilvusClient(uri=MILVUS_URI)
    create_collection(client, dim=dim, fresh=(mode == "build"))

    existing_videos = get_existing_videos(client) if mode == "rebuild" else set()

    total, added, skipped = 0, 0, 0
    print(f"\n--- BẮT ĐẦU NẠP DỮ LIỆU (Mode: {mode}) ---")
    for npy_file in npy_files:
        video_id = os.path.splitext(npy_file)[0]
        if video_id in existing_videos:
            print(f"  - Skip: {video_id} (Đã có sẵn)")
            skipped += 1
            continue
        total += insert_video(client, npy_file)
        added += 1

    # Nạp collection vào bộ nhớ để sẵn sàng search
    client.flush(collection_name=COLLECTION_NAME)
    client.load_collection(collection_name=COLLECTION_NAME)

    print(f"\n Hoàn tất nạp dữ liệu!")
    print(f"Tổng video nạp mới : {added} ({total} vectors)")
    if mode == "rebuild":
        print(f"Tổng video bỏ qua  : {skipped}")

if __name__ == "__main__":
    run(mode="build")

ERROR:grpc._server:Exception calling application: Method not implemented!
Traceback (most recent call last):
  File "c:\conda\Miniconda3\envs\vlm\Lib\site-packages\grpc\_server.py", line 608, in _call_behavior
    response_or_iterator = behavior(argument, context)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\conda\Miniconda3\envs\vlm\Lib\site-packages\pymilvus\grpc_gen\milvus_pb2_grpc.py", line 1264, in AllocTimestamp
    raise NotImplementedError('Method not implemented!')
NotImplementedError: Method not implemented!


[-] Xóa collection cũ: clip_keyframes
[+] Đã tạo collection mới: clip_keyframes (dim=512)

--- BẮT ĐẦU NẠP DỮ LIỆU (Mode: build) ---
  + video_01: Đã nạp 307 vectors
  + video_02: Đã nạp 200 vectors
  + video_03: Đã nạp 120 vectors

 Hoàn tất nạp dữ liệu!
Tổng video nạp mới : 3 (627 vectors)


In [6]:
import numpy as np
from trake_retriever import TrakeEngine

# Truyền rõ đường dẫn file database và collection đã nạp ở Cell 1
engine = TrakeEngine(
    milvus_uri="./milvus_demo.db",
    collection_name="clip_keyframes"
)

# Kiểm tra encode thử 1 câu
emb = engine._encode_text("a man kicking a soccer ball")
print("Embedding shape:", emb.shape)
print("Norm xấp xỉ 1.0:", np.linalg.norm(emb))

c:\conda\Miniconda3\envs\vlm\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Embedding shape: (512,)
Norm xấp xỉ 1.0: 0.99999994


In [ ]:
import zipfile
import io

def validate_submission_zip(zip_path: str):
    print(f"=== BẮT ĐẦU KIỂM TRA FILE: {zip_path} ===")
    errors = []
    warnings = []
    
    if not os.path.exists(zip_path):
        print(f" LỖI: Không tìm thấy file {zip_path}")
        return

    with zipfile.ZipFile(zip_path, 'r') as z:
        file_list = z.namelist()
        
        # 1. Kiểm tra cấu trúc thư mục submission/
        submission_files = [f for f in file_list if f.startswith("submission/") and f.endswith(".csv")]
        if not submission_files:
            errors.append(" File ZIP thiếu thư mục gốc 'submission/' hoặc không chứa file .csv nào bên trong.")
        else:
            print(" Cấu trúc thư mục: Hợp lệ (Có chứa thư mục `submission/`).")

        # 2. Duyệt qua từng file CSV để kiểm tra định dạng
        for csv_file in submission_files:
            print(f"\n--- Đang kiểm tra file: {csv_file} ---")
            raw_content = z.read(csv_file)
            
            # Kiểm tra Encoding UTF-8
            try:
                text_content = raw_content.decode('utf-8')
                print("   Encoding: UTF-8 chuẩn.")
            except UnicodeDecodeError:
                errors.append(f" File {csv_file} không phải định dạng UTF-8.")
                continue

            lines = [l.strip() for l in text_content.splitlines() if l.strip()]
            
            # Kiểm tra số lượng dòng (Tối đa 100 dòng)
            if len(lines) == 0:
                errors.append(f" File {csv_file} bị rỗng (0 dòng).")
                continue
            elif len(lines) > 100:
                errors.append(f" File {csv_file} vượt quá 100 dòng ({len(lines)} dòng).")
            else:
                print(f"   Số dòng dự đoán: {len(lines)}/100 dòng.")

            # Kiểm tra định dạng từng dòng
            for idx, line in enumerate(lines, 1):
                parts = [p.strip() for p in line.split(",")]
                
                # Cột đầu tiên phải là video_id (Không chứa .mp4)
                video_name = parts[0]
                if video_name.lower().endswith(".mp4"):
                    errors.append(f" Dòng {idx} ({csv_file}): Tên video '{video_name}' vẫn còn đuôi .mp4.")
                
                # Các cột sau phải là Frame IDs (Số nguyên)
                frame_strs = parts[1:]
                if len(frame_strs) < 2:
                    errors.append(f" Dòng {idx} ({csv_file}): Chuỗi frame TRAKE quá ngắn ({len(frame_strs)} frames).")
                    continue
                
                frames = []
                for f_str in frame_strs:
                    try:
                        frames.append(int(f_str))
                    except ValueError:
                        errors.append(f" Dòng {idx} ({csv_file}): Frame ID '{f_str}' không phải số nguyên.")
                
                # Kiểm tra thứ tự thời gian tăng dần
                if frames and frames != sorted(frames):
                    warnings.append(f" Dòng {idx} ({csv_file}): Frame IDs không theo thứ tự thời gian tăng dần ({frames}).")

    print("\n================ TỔNG KẾT ================")
    if not errors:
        print(" TẤT CẢ TIÊU CHÍ ĐỀU ĐẠT CHUẨN! File ZIP sẵn sàng nộp lên hệ thống.")
        if warnings:
            for w in warnings:
                print(w)
    else:
        print(f" CÓ {len(errors)} LỖI CẦN SỬA TRƯỚC KHI NỘP:")
        for err in errors:
            print(f"  {err}")

# Chạy kiểm tra
validate_submission_zip("team_test_submission.zip")

=== BẮT ĐẦU KIỂM TRA FILE: team_test_submission.zip ===
❌ LỖI: Không tìm thấy file team_test_submission.zip


In [8]:
import os
import shutil
import zipfile
from aic_agent_core import route_query, TaskType
from trake_retriever import TrakeEngine

# 1. Tạo thư mục và dữ liệu test mẫu
MOCK_DIR = "./queries_test"
SUBMISSION_DIR = "./submission"
os.makedirs(MOCK_DIR, exist_ok=True)
os.makedirs(SUBMISSION_DIR, exist_ok=True)

# Giả lập 2 file query do BTC gửi
test_cases = {
    "query-1-trake.txt": (
        "First is the sunset city skyline with 60 seconds news logo, "
        "then two news anchors male and female presenting in newsroom studio, "
        "after that an aerial drone shot of collapsed asphalt road falling into river water, "
        "and finally a white car driving into hospital entrance."
    ),
    "query-2-trake.txt": (
        "Một vận động viên đang cầm sào chạy đà, "
        "sau đó cắm sào bật nhảy qua xà ngang, "
        "và cuối cùng rơi xuống đệm an toàn."
    )
}

for fname, q_text in test_cases.items():
    with open(os.path.join(MOCK_DIR, fname), "w", encoding="utf-8") as f:
        f.write(q_text)

# 2. Khởi tạo TrakeEngine
engine = TrakeEngine(
    milvus_uri="./milvus_demo.db",
    collection_name="clip_keyframes",
    csv_dir="data/csv_metadata",
    feature_dir="data/clip_features"
)

# 3. Chạy xử lý batch
for filename in os.listdir(MOCK_DIR):
    if not filename.endswith(".txt"):
        continue
    
    query_id = os.path.splitext(filename)[0]
    with open(os.path.join(MOCK_DIR, filename), "r", encoding="utf-8") as f:
        raw_query = f.read().strip()

    task_type, structured_query = route_query(raw_query)
    csv_path = os.path.join(SUBMISSION_DIR, f"{query_id}.csv")
    
    if task_type == TaskType.TRAKE:
        ranked_candidates = engine.solve_trake(structured_query, top_k_videos=100)
        
        with open(csv_path, "w", encoding="utf-8", newline="") as f_out:
            for vid, frames, score in ranked_candidates[:100]:
                frame_str = ", ".join(map(str, frames))
                f_out.write(f"{vid}, {frame_str}\n")
        print(f"[+] Đã tạo: {csv_path} ({len(ranked_candidates[:100])} dòng)")

# 4. Đóng gói ZIP đúng chuẩn cấu trúc BTC yêu cầu
zip_output = "team_test_submission.zip"
with zipfile.ZipFile(zip_output, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(SUBMISSION_DIR):
        for file in files:
            full_p = os.path.join(root, file)
            rel_p = os.path.relpath(full_p, start=".")
            zipf.write(full_p, rel_p)

print(f"\n[OK] Đã hoàn tất tạo file nén: {zip_output}")

[+] Đã tạo: ./submission\query-1-trake.csv (1 dòng)
[+] Đã tạo: ./submission\query-2-trake.csv (1 dòng)

[OK] Đã hoàn tất tạo file nén: team_test_submission.zip


In [ ]:
import zipfile
import io

def validate_submission_zip(zip_path: str):
    print(f"=== BẮT ĐẦU KIỂM TRA FILE: {zip_path} ===")
    errors = []
    warnings = []
    
    if not os.path.exists(zip_path):
        print(f" LỖI: Không tìm thấy file {zip_path}")
        return

    with zipfile.ZipFile(zip_path, 'r') as z:
        file_list = z.namelist()
        
        # 1. Kiểm tra cấu trúc thư mục submission/
        submission_files = [f for f in file_list if f.startswith("submission/") and f.endswith(".csv")]
        if not submission_files:
            errors.append(" File ZIP thiếu thư mục gốc 'submission/' hoặc không chứa file .csv nào bên trong.")
        else:
            print(" Cấu trúc thư mục: Hợp lệ (Có chứa thư mục `submission/`).")

        # 2. Duyệt qua từng file CSV để kiểm tra định dạng
        for csv_file in submission_files:
            print(f"\n--- Đang kiểm tra file: {csv_file} ---")
            raw_content = z.read(csv_file)
            
            # Kiểm tra Encoding UTF-8
            try:
                text_content = raw_content.decode('utf-8')
                print("   Encoding: UTF-8 chuẩn.")
            except UnicodeDecodeError:
                errors.append(f" File {csv_file} không phải định dạng UTF-8.")
                continue

            lines = [l.strip() for l in text_content.splitlines() if l.strip()]
            
            # Kiểm tra số lượng dòng (Tối đa 100 dòng)
            if len(lines) == 0:
                errors.append(f" File {csv_file} bị rỗng (0 dòng).")
                continue
            elif len(lines) > 100:
                errors.append(f" File {csv_file} vượt quá 100 dòng ({len(lines)} dòng).")
            else:
                print(f"   Số dòng dự đoán: {len(lines)}/100 dòng.")

            # Kiểm tra định dạng từng dòng
            for idx, line in enumerate(lines, 1):
                parts = [p.strip() for p in line.split(",")]
                
                # Cột đầu tiên phải là video_id (Không chứa .mp4)
                video_name = parts[0]
                if video_name.lower().endswith(".mp4"):
                    errors.append(f" Dòng {idx} ({csv_file}): Tên video '{video_name}' vẫn còn đuôi .mp4.")
                
                # Các cột sau phải là Frame IDs (Số nguyên)
                frame_strs = parts[1:]
                if len(frame_strs) < 2:
                    errors.append(f" Dòng {idx} ({csv_file}): Chuỗi frame TRAKE quá ngắn ({len(frame_strs)} frames).")
                    continue
                
                frames = []
                for f_str in frame_strs:
                    try:
                        frames.append(int(f_str))
                    except ValueError:
                        errors.append(f" Dòng {idx} ({csv_file}): Frame ID '{f_str}' không phải số nguyên.")
                
                # Kiểm tra thứ tự thời gian tăng dần
                if frames and frames != sorted(frames):
                    warnings.append(f" Dòng {idx} ({csv_file}): Frame IDs không theo thứ tự thời gian tăng dần ({frames}).")

    print("\n================ TỔNG KẾT ================")
    if not errors:
        print(" TẤT CẢ TIÊU CHÍ ĐỀU ĐẠT CHUẨN! File ZIP sẵn sàng nộp lên hệ thống.")
        if warnings:
            for w in warnings:
                print(w)
    else:
        print(f" CÓ {len(errors)} LỖI CẦN SỬA TRƯỚC KHI NỘP:")
        for err in errors:
            print(f"  {err}")

# Chạy kiểm tra
validate_submission_zip("team_test_submission.zip")

=== BẮT ĐẦU KIỂM TRA FILE: team_test_submission.zip ===
✅ Cấu trúc thư mục: Hợp lệ (Có chứa thư mục `submission/`).

--- Đang kiểm tra file: submission/query-1-trake.csv ---
  ✅ Encoding: UTF-8 chuẩn.
  ✅ Số dòng dự đoán: 1/100 dòng.

--- Đang kiểm tra file: submission/query-2-trake.csv ---
  ✅ Encoding: UTF-8 chuẩn.
  ✅ Số dòng dự đoán: 1/100 dòng.

================ TỔNG KẾT ================
🎉 TẤT CẢ TIÊU CHÍ ĐỀU ĐẠT CHUẨN! File ZIP sẵn sàng nộp lên hệ thống.
